In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28,28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
import numpy as np
import matplotlib.pyplot as plt

train_loader = DataLoader(train_dataset, 16, shuffle= True )
test_loader = DataLoader(test_dataset, 16, shuffle= False )


def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(train_loader)
images, labels = next(dataiter)
labels = labels - 1

# Show images
imshow(torchvision.utils.make_grid(images[:4]))
# Print labels
print(' '.join(f'{letters[labels[j]]}' for j in range(4)))


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
from torchvision import models
# Write your code here
# Load pretrained model
model = efficientnet_v2_s(weights = models.EfficientNet_V2_S_Weights)

# Freeze ALL backbone layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head (this will be trainable by default)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 26)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)



In [ ]:
labels - 1

In [ ]:
# Write your code here
# TO DO
def training_loop(model, dataloader, criterion, optimizer, device):
    model.train()  # training mode -> handle dropout and batchnorm
    total_loss = 0
    # we don't only use total loss as this is a classification task

    correct = 0
    total = 0

    for images, labels in tqdm(dataloader): # tqdm to show it as in a loading bar
        images = images.to(device)
        labels = labels - 1
        labels = labels.to(device)
        # print("HELLO")
        # print(labels.shape)

        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # accuracy -> in classification
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  #get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0) # include batches to count the total number of images

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

def validation_loop(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode -> deal with dropout and batchnorm
    total_loss = 0
    # as this is classification task we will include correct and total to get accuracy
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels - 1
            labels = labels.to(device)


            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()


            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
# TO DO
import torch.optim as optim
from tqdm import tqdm

model = model.to(device)
criterion = nn.CrossEntropyLoss() # as this is multiclass classification also without sigmoid (otherwise change the criterion)
optimizer = optim.AdamW(model.parameters(), lr=0.00001)  #fine tue lr to get better results (not always)
num_epochs = 2 # Number of epochs

# Lists to store all the metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = training_loop(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validation_loop(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")



In [ ]:
# plot all of epochs losses
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
def validation_loop(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode -> deal with dropout and batchnorm
    total_loss = 0
    # as this is classification task we will include correct and total to get accuracy
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            h_flipped = torch.flip(images, dims=[3])
            h_outputs = model(h_flipped.to(device))


            v_flipped = torch.flip(images, dims=[2])
            v_outputs = model(v_flipped.to(device))

            image_outputs = model(images.to(device))


            summed_images = h_outputs + v_outputs + image_outputs
            avr = summed_images // 3

            labels = labels - 1
            labels = labels.to(device)

            loss = criterion(avr, labels)  # Compute loss
            total_loss += loss.item()


            avr = torch.softmax(avr, dim=1)
            predictions = avr.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

val_loss, val_accuracy = validation_loop(model, test_loader, criterion, device)


print(f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


